# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate datasets

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

Device: cuda
GPU: Tesla T4
Loading Qwen/Qwen3-4B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
# ============================================================
# Cell 2: RACE loader (JSON, datasets 불필요)
# ============================================================
import random
import re as _re
import json
from urllib.request import urlopen

summarizer = Agent("Summarizer", "You are a Summarizer agent.")
answerer = Agent("Answerer", "You are an Answerer agent.")

print("Loading RACE from GitHub...")
url = "https://raw.githubusercontent.com/Jay931203/wcisl/master/notebooks/data/race_high_test.json"
race_raw = json.loads(urlopen(url).read().decode())

def format_race(ex):
    choices_text = "\n".join(f"{chr(65+i)}) {c}" for i, c in enumerate(ex["options"]))
    return {
        "passage": ex["article"],
        "question": ex["question"],
        "choices_text": choices_text,
        "answer": ex["answer"],
    }

def sample_race(n, seed=42):
    random.seed(seed)
    return [format_race(s) for s in random.sample(race_raw, min(n, len(race_raw)))]

def extract_answer(response):
    text = response.strip()
    if text.upper() in ("A", "B", "C", "D"):
        return text.upper()
    m = _re.search(r'(?:answer|choice)[\s:is]*([A-D])\b', text, _re.IGNORECASE)
    if m: return m.group(1).upper()
    m = _re.search(r'^([A-D])[)\.]', text, _re.MULTILINE)
    if m: return m.group(1)
    m = _re.search(r'\b([A-D])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"RACE loaded: {len(race_raw)} questions")

Loading RACE from GitHub...


HTTPError: HTTP Error 404: Not Found

---
# 핵심 아이디어 1: Tx 인지 수준별 통신 효율 (RACE 20문제, 4조건)
- **A (Summarizer)**: 지문 보고 요약 전송 (중요 정보 먼저)
- **B (Answerer)**: 요약+질문+선택지로 답

| 조건 | A sees | B prompt |
|------|--------|----------|
| blind | passage only | 일반 |
| a_aware | passage + choices | 일반 |
| b_aware | passage only | A 인지 |
| mutual | passage + choices | A 인지 |

- TX 예산: 32 / 64 / 128 tokens
- A 캐싱: blind/choices 각 1회만 생성 → b_aware/mutual 재사용

In [ ]:
# ============================================================
# 핵심 아이디어 1: 4조건 (blind / a_aware / b_aware / mutual)
# A 캐싱: blind/choices 각 budget당 1회만 생성
# ============================================================

A_BLIND = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. Start with the most important fact. "
    "Capture the most important facts and events."
)
A_CHOICES = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader must choose between the answer options shown below. "
    "Start with the ONE fact that best distinguishes between the options. "
    "Do NOT indicate which option is correct."
)

B_BLIND = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)
B_AWARE = (
    "You are an Answerer. You receive a summary from a Summarizer agent "
    "who saw the answer choices and tailored the summary to highlight "
    "distinctions between the options. "
    "The summary is focused on choice-relevant details \u2014 use it with confidence. "
    "Output ONLY a single letter: A, B, C, or D."
)

CONDITIONS = {
    "blind":   {"a_type": "blind",   "b_knows": False},
    "a_aware": {"a_type": "choices", "b_knows": False},
    "b_aware": {"a_type": "blind",   "b_knows": True},
    "mutual":  {"a_type": "choices", "b_knows": True},
}

TX_BUDGETS = [32, 64, 128]

Q1 = sample_race(20, seed=42)
print(f"RACE 4-COND: {len(Q1)} questions, budgets={TX_BUDGETS}")

# --- Run ---
results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

for qi, q in enumerate(Q1):
    print(f"\n{'='*60}")
    print(f"Q{qi+1}: {q['question'][:60]}...")
    print(f"Expected: {q['answer']}")

    for budget in TX_BUDGETS:
        # Cache A outputs (blind and choices, once per budget)
        a_cache = {}
        summarizer.set_prompt(A_BLIND)
        a_cache["blind"] = summarizer.say(
            f"Passage:\n{q['passage']}", max_tokens=budget
        )
        print(f"  [A-blind @{budget}tok]: {a_cache['blind']['response'][:150]}")

        summarizer.set_prompt(A_CHOICES)
        a_cache["choices"] = summarizer.say(
            f"Passage:\n{q['passage']}\n\nAnswer options:\n{q['choices_text']}",
            max_tokens=budget,
        )
        print(f"  [A-choices @{budget}tok]: {a_cache['choices']['response'][:150]}")

        for cond, cfg in CONDITIONS.items():
            a_out = a_cache[cfg["a_type"]]

            # B
            answerer.set_prompt(B_AWARE if cfg["b_knows"] else B_BLIND)
            b_out = answerer.say(
                f"Summary:\n{a_out['response']}\n\n"
                f"Question: {q['question']}\n"
                f"Choices:\n{q['choices_text']}\n\nAnswer:",
                max_tokens=48,
            )
            pred = extract_answer(b_out["response"])
            correct = pred == q["answer"]
            mark = "\u2713" if correct else "\u2717"
            print(f"  [{cond:14s}] B='{b_out['response'][:50]}' \u2192 {pred} {mark}")

            results[budget][cond].append({
                "correct": correct,
                "answer": pred,
                "expected": q["answer"],
                "a_tokens": a_out["tokens"],
            })

# Restore
summarizer.set_prompt("You are a Summarizer agent.")
answerer.set_prompt("You are an Answerer agent.")

# --- Results table ---
print(f"\n{'Budget':<8} {'blind':<10} {'a_aware':<10} {'b_aware':<10} {'mutual':<10}")
print("-" * 48)
for budget in TX_BUDGETS:
    row = f"{budget}tok  "
    for cond in CONDITIONS:
        res = results[budget][cond]
        acc = sum(r["correct"] for r in res) / len(res)
        row += f"{acc*100:>5.0f}%    "
    print(row)

# Rate-distortion curve
print(f"\n--- Rate-Distortion ---")
for cond in CONDITIONS:
    points = " \u2192 ".join(
        f"{b}({sum(r['correct'] for r in results[b][cond])/len(results[b][cond])*100:.0f}%)"
        for b in TX_BUDGETS
    )
    print(f"  {cond:<10}: {points}")

---
# 핵심 아이디어 2: 단계별 CoT 전환 (RACE 20문제)
- **A**: 3단계 CoT (이해→추론→전달), Stage 3 출력만 B에게 전송
- **B**: Stage 3 출력 + 질문 + 선택지로 답

| 조건 | Stage 1-2 | Stage 3 | B |
|------|:-:|:-:|:-:|
| all_general | 일반 | 일반 | 일반 |
| all_aware | task 인지 | task 인지 | A 과정 인지 |
| tx_switch | 일반 | task 인지 | 일반 |
| both_switch | 일반 | task 인지 | A 과정 인지 |

In [ ]:
# ============================================================
# 핵심 아이디어 2: 단계별 CoT 전환 (RACE 20문제)
# ============================================================

STAGE_GEN = [
    "Read this passage and identify the main topic, key events, and important details.\n\nPassage:\n{passage}",
    "Based on your analysis, determine which facts are most important. Focus on concrete details.\n\n{prev}",
    "Write a concise message for another agent who cannot see the passage. Keep it to 2-3 sentences.\n\n{prev}",
]

STAGE_AWR = [
    "Read this passage and identify information relevant for answering a reading comprehension question.\n\nPassage:\n{passage}",
    "Determine which facts would help distinguish between answer choices. Focus on task-relevant details.\n\n{prev}",
    "Write a concise message for an agent answering a reading comprehension question. Start with the most distinguishing fact. Do NOT state the answer. Keep to 2-3 sentences.\n\n{prev}",
]

B_BLIND_K2 = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)

B_AWARE_K2 = (
    "You are an Answerer. You receive a summary from a Summarizer agent.\n\n"
    "About the Summarizer's process:\n"
    "- Stage 1-2: general analysis (did NOT know your question)\n"
    "- Stage 3: knew about the task and refined the message\n\n"
    "Interpret accordingly:\n"
    "- Details in the summary were intentionally selected in Stage 3\n"
    "- Trust task-relevant cues \u2014 they were chosen on purpose\n\n"
    "Output ONLY a single letter: A, B, C, or D."
)

COT_CONDITIONS = {
    "all_general":  {"stage_aware": [False, False, False], "b_aware": False},
    "all_aware":    {"stage_aware": [True, True, True],    "b_aware": True},
    "tx_switch":    {"stage_aware": [False, False, True],  "b_aware": False},
    "both_switch":  {"stage_aware": [False, False, True],  "b_aware": True},
}

Q2 = sample_race(20, seed=99)
print(f"Key Idea 2: {len(Q2)} RACE questions (max_tokens=128 for all stages)")

# --- Run ---
k2_results = {c: [] for c in COT_CONDITIONS}

for qi, q in enumerate(Q2):
    print(f"\n{'='*60}")
    print(f"Q{qi+1}: {q['question'][:60]}...")
    print(f"Expected: {q['answer']}")

    # Cache A outputs by stage_aware config
    a_cache = {}

    for cond_name, cfg in COT_CONDITIONS.items():
        stage_key = str(cfg["stage_aware"])

        if stage_key not in a_cache:
            # Run 3-stage CoT
            prev = ""
            for i, aware in enumerate(cfg["stage_aware"]):
                stages = STAGE_AWR if aware else STAGE_GEN
                msg = stages[i].format(passage=q["passage"], prev=prev)
                summarizer.set_prompt("You are a Summarizer agent.")
                r = summarizer.say(msg, max_tokens=128)
                tag = "aware" if aware else "general"
                print(f"  [A-Stage{i+1} ({tag})]: {r['response'][:150]}")
                prev = r["response"]
            a_cache[stage_key] = prev

        a_msg = a_cache[stage_key]

        # B
        b_sys = B_AWARE_K2 if cfg["b_aware"] else B_BLIND_K2
        answerer.set_prompt(b_sys)
        b_out = answerer.say(
            f"Summary:\n{a_msg}\n\n"
            f"Question: {q['question']}\n"
            f"Choices:\n{q['choices_text']}\n\nAnswer:",
            max_tokens=48,
        )
        pred = extract_answer(b_out["response"])
        correct = pred == q["answer"]
        mark = "\u2713" if correct else "\u2717"
        print(f"  [{cond_name:14s}] B='{b_out['response'][:50]}' \u2192 {pred} {mark}")

        k2_results[cond_name].append({
            "correct": correct,
            "answer": pred,
            "expected": q["answer"],
        })

# Restore
summarizer.set_prompt("You are a Summarizer agent.")
answerer.set_prompt("You are an Answerer agent.")

# --- Results table ---
print(f"\n{'Condition':<16} {'Accuracy':<10}")
print("-" * 26)
for cond in COT_CONDITIONS:
    res = k2_results[cond]
    acc = sum(r["correct"] for r in res) / len(res)
    print(f"{cond:<16} {acc*100:>5.0f}%")

# Key comparison
print(f"\n--- \ud575\uc2ec \ube44\uad50 ---")
gen = sum(r["correct"] for r in k2_results["all_general"]) / len(k2_results["all_general"])
sw = sum(r["correct"] for r in k2_results["tx_switch"]) / len(k2_results["tx_switch"])
both = sum(r["correct"] for r in k2_results["both_switch"]) / len(k2_results["both_switch"])
full = sum(r["correct"] for r in k2_results["all_aware"]) / len(k2_results["all_aware"])
print(f"  all_general {gen*100:.0f}% \u2192 tx_switch {sw*100:.0f}% \u2192 both_switch {both*100:.0f}% \u2192 all_aware {full*100:.0f}%")